## 前提条件
- 設計方針は、instruction → 正規化意味表現 → 実行プラン → 編集実行 の4段に分けるのがよいです。
- ver01 / ver02 は instruction の表現ゆれ、ver08 は instruction に対する GT task であり、<br>この組み合わせは「表現差に頑健な instruction parser」を作るのに適しています。

## 1. 全体方針
- 最初から instruction から OpenCV / VACE / PISCO などの具体手段を直接出させるより、まず 中間表現 を作るべきです。
- 理由は、ver08 の task はかなり有用ですが、target や params の粒度に揺れや壊れた例も一部あり、<br>
  そのまま最終実行形式にすると downstream が不安定になるためです。
- たとえば色変更で target が長文化している例や、replacement の category が対象全体ではなく hat に寄っている例が見られます。

- したがって、LLM の責務は次の2つに限定します。
    1. instruction を読んで 意味を壊さずに構造化 する
    2. 構造化結果から 後段が選択可能な編集意図 を出す

- 編集そのものの品質は後段の CV / video editing pipeline に寄せます。
- この分離をしないと、LLM の気分で処理系の選択が揺れます。

## 2. 推奨アーキテクチャ
### A. Instruction Parser 層

入力:
- ver01 / ver02 の instruction
- 必要なら video_path
- 必要なら selected_class / selected_subclass 相当の補助情報

出力:
- Canonical Edit Spec（正規化中間表現）

ここでは、表現ゆれを吸収します。

「Replace」「Swap」「Edit the video so that replace」のような言い換え差を同じ意味に畳みます。

ver01 / ver02 はまさにこの学習材料です。

### B. Planner 層

入力:
- Canonical Edit Spec

出力:
- 実行可能な subtask 列
    - 必要な検出対象
    - 必要な補助処理
    - 推奨 backend

ここで初めて、OpenCV / GroundingDINO+SAM / PISCO / LaMa / E2FGVI / RAFT / diffusion 系を選びます。

各編集カテゴリごとの現実解は、添付の詳細版 md 群にかなり整理されています。

背景変更は 
- segmentation 中心、
- 色変更は OpenCV 中心、
- 数量増加は insertion 問題、
- instance replacement は SAM + diffusion / PISCO 系、
- motion は pose or tracking、
- camera motion は zoom と dolly を分ける、
という設計が妥当です。

### C. Executor 層

入力:
- planner 出力
- 動画フレーム列

出力:
- 編集済み動画

ここは LLM を使わず、決め打ちの実装にします。

In [1]:
print("test")

test
